# CS781 Phase 3: Rhetorical Role labelling with a LORMA-adapted decoder-only LM

Pipeline in this notebook (single GPU, no subprocess / DDP, so there are fewer moving parts):

1. Load the **Phase 1 pretrained decoder** (`best_model.pt`) and the Phase 1 tokenizer from the attached input.
2. Wrap it with **LORMA adapters** (SVD-initialised U, Sigma, V on the Q and V projections) plus a **sentence classifier**. The base model is frozen.
3. Every document is split into **sentence-aligned windows** of at most 2048 tokens, with a little left context, so **no sentence is ever dropped** (the old notebook cut every document at 2048 tokens).
4. Train on the windows, pick the epoch with the best validation macro-F1.
5. Predict **every sentence of every test document**, then run an independent validator on the saved zip before you submit it.

Attach the Phase 1 output (`cs781_phase1`: `checkpoints/best_model.pt`, `legal_bpe.json`) as a notebook input, use a GPU accelerator, and Run All.

In [ ]:
import os, sys, json, glob, math, time, random, zipfile, itertools
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
else:
    print("WARNING: no Hugging Face token found (Kaggle secret 'HF' or env HF_TOKEN). Downloads may fail.")

print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Enable a GPU accelerator (Session options -> Accelerator)."
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))
device = torch.device("cuda:0")      # training / inference use a single GPU
use_amp = True

In [ ]:
# =============================================================================
# Configuration
# =============================================================================
SEED = 42

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_FILES = {
    "train": ["Phase_3_RR/Train/CL_train.json", "Phase_3_RR/Train/IT_train.json"],
    "dev":   ["Phase_3_RR/Dev/CL_dev.json", "Phase_3_RR/Dev/IT_dev.json"],     # optional (404 at the moment)
    "test":  ["Phase_3_RR/Test/CL_test.json", "Phase_3_RR/Test/IT_test.json"],
    "vocab": "Phase_3_RR/Train/label_vocab.json",
}

WORK = "/kaggle/working/cs781_phase3"
DATA_LOCAL = os.path.join(WORK, "data")
BEST_STATE_PATH = os.path.join(WORK, "rr_lorma_best_state.pt")
PREDICTION_PATH = os.path.join(WORK, "prediction.json")
SUBMISSION_ZIP = os.path.join(WORK, "submission.zip")
os.makedirs(DATA_LOCAL, exist_ok=True)

# Must equal the Phase 1 architecture exactly (taken from the Phase 1 train_config.json)
MODEL_KWARGS = dict(vocab_size=32000, d_model=768, n_heads=12, n_layers=10,
                    max_seq_len=2048, dropout=0.1, bias=True)

# Optional: force a specific Phase 1 checkpoint instead of auto-discovery
PHASE1_CKPT_OVERRIDE = None

# ---- windowing: how documents are cut so that every sentence is covered -------
MAX_LEN = 2048        # tokens per window (= Phase 1 context length)
SENT_CAP = 512        # a single sentence longer than this is cut to its first 512 tokens
CTX_TOKENS = 384      # up to this many tokens of preceding sentences are re-read as context
assert SENT_CAP + CTX_TOKENS < MAX_LEN

# ---- LORMA / head ---------------------------------------------------------------
LORMA_RANK = 16
LORMA_ALPHA = 16.0
LORMA_L1 = 1e-4                 # L1 penalty on the Sigma vectors
SUBTRACT_PRINCIPAL = True       # True: frozen weight = W - (top-r part), so the model starts exactly at the pretrained function
USE_POSITION_FEATURES = True    # relative sentence position (i / (n-1)) fed to the classifier head

# ---- training --------------------------------------------------------------------
EPOCHS = 15
MICRO_BS = 2                    # windows per forward/backward (lower to 1 if you ever hit CUDA OOM)
GLOBAL_BS = 8                   # windows per optimizer step
LR_LORMA = 5e-4
LR_HEAD = 2e-3
WD_HEAD = 0.01
WARMUP_FRAC = 0.1
VAL_FRACTION = 0.2              # only used when no Dev files are available
CLASS_WEIGHT_POWER = 0.5        # class weight = (1/count)^power, normalised to mean 1 (0 = unweighted)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()
print("Configuration loaded.")

In [ ]:
# =============================================================================
# Locate the Phase 1 tokenizer and pretrained checkpoint in /kaggle/input
# =============================================================================
def glob_input(pattern):
    return sorted(glob.glob(os.path.join("/kaggle/input", "**", pattern), recursive=True))

def prefer_phase1(paths):
    return sorted(paths, key=lambda p: (0 if "cs781_phase1" in p else 1, p))

# ---- tokenizer
tok_candidates = prefer_phase1(glob_input("legal_bpe.json"))
assert tok_candidates, "legal_bpe.json not found under /kaggle/input (attach the Phase 1 output)."
TOKENIZER_PATH = tok_candidates[0]

from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
assert tokenizer.get_vocab_size() == MODEL_KWARGS["vocab_size"], tokenizer.get_vocab_size()
print("Tokenizer:", TOKENIZER_PATH, "| vocab", tokenizer.get_vocab_size())

# ---- checkpoint
def extract_state_dict(ckpt):
    if isinstance(ckpt, dict):
        for k in ("model_state", "model_state_dict", "model", "state_dict"):
            if k in ckpt and isinstance(ckpt[k], dict):
                return ckpt[k]
        if ckpt and all(torch.is_tensor(v) for v in ckpt.values()):
            return ckpt
    return None

def clean_keys(sd):
    out = {}
    for k, v in sd.items():
        for pre in ("module.", "_orig_mod."):
            while k.startswith(pre):
                k = k[len(pre):]
        out[k] = v
    return out

def is_phase1_state(sd):
    return ("transformer.wte.weight" in sd
            and "transformer.h.0.attn.c_attn.weight" in sd
            and tuple(sd["transformer.wte.weight"].shape) == (MODEL_KWARGS["vocab_size"], MODEL_KWARGS["d_model"]))

ckpt_candidates = [PHASE1_CKPT_OVERRIDE] if PHASE1_CKPT_OVERRIDE else prefer_phase1(glob_input("best_model.pt"))
print("Checkpoint candidates:", ckpt_candidates)

PHASE1_STATE, PHASE1_CKPT_PATH = None, None
for p in ckpt_candidates:
    try:
        raw = torch.load(p, map_location="cpu", weights_only=False)
        sd = extract_state_dict(raw)
        if sd is None:
            print("  rejected (no state dict found):", p); continue
        sd = clean_keys(sd)
        if not is_phase1_state(sd):
            print("  rejected (not a Phase 1 decoder checkpoint):", p); continue
        PHASE1_STATE, PHASE1_CKPT_PATH = sd, p
        if isinstance(raw, dict):
            print("  checkpoint metadata:", {k: v for k, v in raw.items() if isinstance(v, (int, float, str))})
        del raw
        break
    except Exception as e:
        print("  could not read", p, "->", type(e).__name__, e)

assert PHASE1_STATE is not None, "No usable Phase 1 checkpoint found. Attach the Phase 1 output or set PHASE1_CKPT_OVERRIDE."
print("Using Phase 1 checkpoint:", PHASE1_CKPT_PATH, "|", len(PHASE1_STATE), "tensors")

In [ ]:
# =============================================================================
# Download the Phase 3 data
# =============================================================================
from huggingface_hub import hf_hub_download

def fetch(rel, required=True):
    try:
        p = hf_hub_download(repo_id=HF_DATASET_REPO, filename=rel, repo_type="dataset",
                            token=HF_TOKEN, local_dir=DATA_LOCAL)
        print("  [+]", rel)
        return p
    except Exception as e:
        if required:
            raise RuntimeError(f"Could not download required file {rel}: {type(e).__name__}: {e}")
        print("  [-] not available:", rel)
        return None

TRAIN_PATHS = [fetch(r) for r in HF_FILES["train"]]
TEST_PATHS  = [fetch(r) for r in HF_FILES["test"]]
VOCAB_PATH  = fetch(HF_FILES["vocab"])
DEV_PATHS   = [p for p in (fetch(r, required=False) for r in HF_FILES["dev"]) if p]

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    label_to_id = {str(k): int(v) for k, v in json.load(f).items()}
assert sorted(label_to_id.values()) == list(range(len(label_to_id))), "label ids are not 0..N-1"
id_to_label = {v: k for k, v in label_to_id.items()}
NUM_LABELS = len(label_to_id)
print(f"\n{NUM_LABELS} labels:", label_to_id)

In [ ]:
# =============================================================================
# Parsing, tokenisation and sentence-aligned windows
# =============================================================================
ID_KEYS = ("id", "doc_id", "docid", "name", "file", "filename")
TEXT_KEYS = ("text", "sentences", "sents", "sentence_list")
LABEL_KEYS = ("labels", "label", "roles")

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def iter_records(data, source):
    # Yield (doc_id, record) from {id: record} or [record, ...]
    if isinstance(data, dict):
        for k, v in data.items():
            yield str(k), v
    elif isinstance(data, list):
        for i, rec in enumerate(data):
            if not isinstance(rec, dict):
                raise ValueError(f"{source}: record {i} is a {type(rec).__name__}, expected an object with an id")
            did = next((rec[k] for k in ID_KEYS if k in rec and rec[k] is not None), None)
            if did is None:
                raise ValueError(f"{source}: record {i} has no id field (keys: {list(rec.keys())})")
            yield str(did), rec
    else:
        raise ValueError(f"{source}: unexpected top-level type {type(data).__name__}")

def get_sentences(rec, doc_id):
    if isinstance(rec, list):
        sents = rec
    elif isinstance(rec, dict):
        key = next((k for k in TEXT_KEYS if k in rec), None)
        if key is None:
            raise ValueError(f"document {doc_id}: no sentence list found (keys: {list(rec.keys())})")
        sents = rec[key]
    else:
        raise ValueError(f"document {doc_id}: unexpected record type {type(rec).__name__}")
    if not isinstance(sents, list) or not all(isinstance(s, str) for s in sents):
        raise ValueError(f"document {doc_id}: sentences must be a list of strings")
    return sents

def get_labels(rec):
    if isinstance(rec, dict):
        for k in LABEL_KEYS:
            if k in rec and isinstance(rec[k], list):
                return rec[k]
    return None

def to_label_id(x):
    if isinstance(x, str):
        if x not in label_to_id:
            raise ValueError(f"label {x!r} is not in label_vocab.json")
        return label_to_id[x]
    x = int(x)
    if not 0 <= x < NUM_LABELS:
        raise ValueError(f"label id {x} out of range")
    return x

_dot = tokenizer.encode(".", add_special_tokens=False).ids
FALLBACK_ID = _dot[0] if _dot else 0

def tokenize_sentences(sentences):
    # One token list per sentence (never empty, never longer than SENT_CAP).
    out, n_trunc = [], 0
    for i, s in enumerate(sentences):
        text = s if i == 0 else " " + s          # keep the word-boundary space a continuous text would have
        ids = tokenizer.encode(text, add_special_tokens=False).ids
        if not ids:
            ids = [FALLBACK_ID]                  # empty / whitespace-only sentence still needs a position to predict from
        if len(ids) > SENT_CAP:
            ids = ids[:SENT_CAP]; n_trunc += 1
        out.append(ids)
    return out, n_trunc

def load_split(paths, labeled, name, strict_ids):
    docs, skipped, seen = [], [], {}
    for path in paths:
        for doc_id, rec in iter_records(load_json(path), path):
            sents = get_sentences(rec, doc_id)
            labs = None
            if labeled:
                raw_labs = get_labels(rec)
                if raw_labs is None or len(raw_labs) != len(sents):
                    skipped.append(doc_id); continue
                labs = [to_label_id(x) for x in raw_labs]
            if doc_id in seen:
                if strict_ids:
                    raise ValueError(f"duplicate document id in {name}: {doc_id}")
                seen[doc_id] += 1; doc_id = f"{doc_id}__dup{seen[doc_id]}"
            seen[doc_id] = 0
            toks, n_trunc = tokenize_sentences(sents)
            docs.append(dict(id=doc_id, n_sent=len(sents), sent_tok=toks, labels=labs, n_trunc=n_trunc))
    if skipped:
        print(f"  [{name}] skipped {len(skipped)} doc(s) with missing/misaligned labels: {skipped[:5]}")
    return docs

def window_spans(lens):
    # (context_start, first_new, end) per window. New sentences [first_new, end) are predicted; [context_start, first_new) is context only.
    n, spans, start = len(lens), [], 0
    while start < n:
        c, ctx = start, 0
        while c > 0 and ctx + lens[c - 1] <= CTX_TOKENS:
            c -= 1
            ctx += lens[c]
        end, total = start, ctx
        while end < n and total + lens[end] <= MAX_LEN:
            total += lens[end]
            end += 1
        assert end > start, "window could not advance"
        spans.append((c, start, end))
        start = end
    return spans

def build_windows(doc, use_labels):
    toks = doc["sent_tok"]; n = len(toks); lens = [len(t) for t in toks]
    out = []
    for c, s, e in window_spans(lens):
        ids = [t for sent in toks[c:e] for t in sent]
        ends = [x - 1 for x in itertools.accumulate(lens[c:e])]
        gidx = list(range(c, e))
        is_new = [g >= s for g in gidx]
        rel = [g / (n - 1) if n > 1 else 0.0 for g in gidx]
        labs = [doc["labels"][g] if (use_labels and nw) else -1 for g, nw in zip(gidx, is_new)]
        out.append(dict(input_ids=torch.tensor(ids, dtype=torch.long),
                        end_idx=torch.tensor(ends, dtype=torch.long),
                        rel=torch.tensor(rel, dtype=torch.float32),
                        labels=torch.tensor(labs, dtype=torch.long),
                        is_new=torch.tensor(is_new, dtype=torch.bool),
                        sent_idx=torch.tensor(gidx, dtype=torch.long)))
    return out

def verify_windows(docs, name):
    # Every sentence must be a 'new' sentence in exactly one window, and every window must fit the context.
    for d in docs:
        lens = [len(t) for t in d["sent_tok"]]
        covered = []
        for c, s, e in window_spans(lens):
            assert sum(lens[c:e]) <= MAX_LEN, (d["id"], "window too long")
            covered.extend(range(s, e))
        assert covered == list(range(len(lens))), f"{name}: sentence coverage broken for {d['id']}"
    print(f"  [{name}] window coverage verified for {len(docs)} docs")

def describe(docs, name):
    n_sent = sum(d["n_sent"] for d in docs)
    n_tok = sum(sum(len(t) for t in d["sent_tok"]) for d in docs)
    long_docs = sum(1 for d in docs if sum(len(t) for t in d["sent_tok"]) > MAX_LEN)
    n_win = sum(len(window_spans([len(t) for t in d["sent_tok"]])) for d in docs)
    trunc = sum(d["n_trunc"] for d in docs)
    print(f"  [{name}] {len(docs)} docs | {n_sent} sentences | {n_tok} tokens | {long_docs} docs exceed {MAX_LEN} tokens"
          f" | {n_win} windows | {trunc} over-long sentences cut to {SENT_CAP} tokens")

print("Tokenising ...")
all_train_docs = load_split(TRAIN_PATHS, True, "train", strict_ids=False)
test_docs      = load_split(TEST_PATHS, False, "test", strict_ids=True)
dev_docs       = load_split(DEV_PATHS, True, "dev", strict_ids=False) if DEV_PATHS else []

if dev_docs:
    train_docs, val_docs = all_train_docs, dev_docs
    print("Validation set: official Dev files")
else:
    rng = random.Random(SEED)
    order = list(range(len(all_train_docs))); rng.shuffle(order)
    n_val = max(1, int(round(len(order) * VAL_FRACTION)))
    val_docs = [all_train_docs[i] for i in order[:n_val]]
    train_docs = [all_train_docs[i] for i in order[n_val:]]
    print(f"Validation set: {n_val} documents held out from the training files (no Dev files available)")

print()
for dset, nm in ((train_docs, "train"), (val_docs, "val"), (test_docs, "test")):
    describe(dset, nm)
    verify_windows(dset, nm)

# class weights from the training sentences
counts = np.zeros(NUM_LABELS)
for d in train_docs:
    for l in d["labels"]:
        counts[l] += 1
print("\nTrain label counts:", {id_to_label[i]: int(c) for i, c in enumerate(counts)})
if CLASS_WEIGHT_POWER > 0:
    w = (1.0 / np.maximum(counts, 1.0)) ** CLASS_WEIGHT_POWER
    w = w / w.mean()
else:
    w = np.ones(NUM_LABELS)
class_weights = torch.tensor(w, dtype=torch.float32)

In [ ]:
# =============================================================================
# Model: Phase 1 decoder (same module names as model_dt.py) + LORMA + classifier head
# =============================================================================
from dataclasses import dataclass

NEG_INF = -65504.0

@dataclass
class LegalLMConfig:
    vocab_size: int = 32000
    d_model: int = 768
    n_heads: int = 12
    n_layers: int = 10
    max_seq_len: int = 2048
    dropout: float = 0.1
    bias: bool = True

def compute_alibi_slopes(n_heads: int) -> torch.Tensor:
    n = 2 ** math.floor(math.log2(n_heads))
    base = 2 ** (-(2 ** -(math.log2(n) - 3)))
    slopes = [base * (base ** i) for i in range(n)]
    if n < n_heads:
        extra_base = 2 ** (-(2 ** -(math.log2(2 * n) - 3)))
        slopes.extend([extra_base ** (2 * i + 1) for i in range(n_heads - n)])
    return torch.tensor(slopes, dtype=torch.float32)

class ALiBiCausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.d_model % config.n_heads == 0
        self.n_heads = config.n_heads
        self.d_head = config.d_model // config.n_heads
        self.dropout_p = config.dropout
        self.c_attn = nn.Linear(config.d_model, 3 * config.d_model, bias=config.bias)   # fused Q,K,V
        self.c_proj = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x, attn_bias, past_kv=None, use_cache=False):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present = (k, v) if use_cache else None
        bias = attn_bias.get(q.dtype)
        if bias is None:
            bias = attn_bias[q.dtype] = attn_bias[torch.float32].to(q.dtype)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=bias,
                                           dropout_p=self.dropout_p if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y, present

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, 4 * config.d_model, bias=config.bias),
            nn.GELU(),
            nn.Linear(4 * config.d_model, config.d_model, bias=config.bias),
            nn.Dropout(config.dropout),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = nn.LayerNorm(config.d_model)
        self.attn = ALiBiCausalSelfAttention(config)
        self.ln2 = nn.LayerNorm(config.d_model)
        self.mlp = FeedForward(config)
    def forward(self, x, attn_bias, past_kv=None, use_cache=False):
        a, present = self.attn(self.ln1(x), attn_bias, past_kv, use_cache)
        x = x + a
        x = x + self.mlp(self.ln2(x))
        return x, present

class CustomLegalLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte=nn.Embedding(config.vocab_size, config.d_model),
            drop=nn.Dropout(config.dropout),
            h=nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)]),
            ln_f=nn.LayerNorm(config.d_model),
        ))
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight          # weight tying (as in Phase 1)
        self.register_buffer("alibi_slopes", compute_alibi_slopes(config.n_heads).view(1, config.n_heads, 1, 1),
                             persistent=False)
        self._bias_key, self._bias_val = None, None

    def _alibi_bias(self, T, S, device):
        key = (T, S, str(device))
        if self._bias_key == key:
            return self._bias_val
        q_pos = torch.arange(S - T, S, device=device).view(T, 1)
        k_pos = torch.arange(S, device=device).view(1, S)
        dist = (q_pos - k_pos).view(1, 1, T, S)
        bias = -self.alibi_slopes.to(device).float() * dist.clamp(min=0).float()
        bias = bias.masked_fill(dist < 0, NEG_INF)
        self._bias_key, self._bias_val = key, {torch.float32: bias}
        return self._bias_val

    def _run(self, input_ids, past_kvs=None, use_cache=False):
        B, T = input_ids.size()
        past_len = 0 if past_kvs is None else past_kvs[0][0].size(2)
        bias = self._alibi_bias(T, past_len + T, input_ids.device)
        x = self.transformer.drop(self.transformer.wte(input_ids))
        presents = []
        for i, block in enumerate(self.transformer.h):
            x, present = block(x, bias, None if past_kvs is None else past_kvs[i], use_cache)
            presents.append(present)
        return self.transformer.ln_f(x), presents

    def forward(self, input_ids, targets=None):
        x, _ = self._run(input_ids)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1))
        else:
            logits = self.lm_head(x[:, [-1], :]); loss = None
        return logits, loss

# ------------------------------------------------------------------------------
# LORMA on the fused QKV projection (adapters on Q and V only)
# ------------------------------------------------------------------------------
class LORMAQKV(nn.Module):
    # Replaces attn.c_attn. For the Q and V row-blocks of the weight, the top-`rank` SVD components
    # (U, Sigma, V^T) become trainable parameters. If subtract_principal is True, those components are
    # removed from the frozen weight, so at initialisation the layer reproduces the pretrained output exactly.
    def __init__(self, c_attn, d_model, rank, alpha, subtract_principal=True):
        super().__init__()
        d = d_model
        self.scaling = alpha / rank
        W = c_attn.weight.detach().double().cpu()           # (3d, d)
        W_res = W.clone()
        for tag, sl in (("q", slice(0, d)), ("v", slice(2 * d, 3 * d))):
            U, S, Vh = torch.linalg.svd(W[sl], full_matrices=False)
            U_r, S_r, Vh_r = U[:, :rank], S[:rank], Vh[:rank, :]
            if subtract_principal:
                W_res[sl] -= self.scaling * (U_r * S_r) @ Vh_r
            setattr(self, f"{tag}_U", nn.Parameter(U_r.float().contiguous()))
            setattr(self, f"{tag}_S", nn.Parameter(S_r.float().contiguous()))
            setattr(self, f"{tag}_Vt", nn.Parameter(Vh_r.float().contiguous()))
        self.register_buffer("W_res", W_res.float())
        bias = c_attn.bias
        self.register_buffer("b", None if bias is None else bias.detach().cpu().float().clone())

    def _delta(self, x, tag):
        h = F.linear(x, getattr(self, f"{tag}_Vt"))
        h = h * getattr(self, f"{tag}_S").to(h.dtype)
        return F.linear(h, getattr(self, f"{tag}_U")) * self.scaling

    def forward(self, x):
        out = F.linear(x, self.W_res, self.b)
        q, k, v = out.chunk(3, dim=-1)
        return torch.cat([q + self._delta(x, "q").to(q.dtype), k, v + self._delta(x, "v").to(v.dtype)], dim=-1)

    def sigma_l1(self):
        return self.q_S.abs().sum() + self.v_S.abs().sum()

def lorma_l1(model):
    return sum(m.sigma_l1() for m in model.modules() if isinstance(m, LORMAQKV))

class DecoderLORMAForRR(nn.Module):
    def __init__(self, base, num_labels, rank, alpha, subtract_principal=True, use_pos=True):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)                                  # the pretrained decoder is frozen
        d = base.config.d_model
        for block in self.base.transformer.h:
            block.attn.c_attn = LORMAQKV(block.attn.c_attn, d, rank, alpha, subtract_principal)
        self.use_pos = use_pos
        self.drop = nn.Dropout(0.1)
        self.classifier = nn.Linear(d + (5 if use_pos else 0), num_labels)
        nn.init.normal_(self.classifier.weight, mean=0.0, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def train(self, mode=True):
        super().train(mode)
        self.base.train(False)                                       # no dropout inside the frozen decoder
        return self

    @staticmethod
    def _pos_feats(rel):
        r = rel.unsqueeze(-1)
        return torch.cat([r, torch.sin(math.pi * r), torch.cos(math.pi * r),
                          torch.sin(2 * math.pi * r), torch.cos(2 * math.pi * r)], dim=-1)

    def forward(self, input_ids, end_idx, rel):
        h, _ = self.base._run(input_ids)                             # (B, T, d)
        rep = h.gather(1, end_idx.unsqueeze(-1).expand(-1, -1, h.size(-1)))   # hidden state at each sentence's last token
        if self.use_pos:
            rep = torch.cat([rep, self._pos_feats(rel).to(rep.dtype)], dim=-1)
        return self.classifier(self.drop(rep))                       # (B, S, num_labels)

print("Model classes defined.")

In [ ]:
# =============================================================================
# Load the pretrained weights and verify that the load really worked
# =============================================================================
cfg = LegalLMConfig(**MODEL_KWARGS)
base = CustomLegalLM(cfg)

missing, unexpected = base.load_state_dict(PHASE1_STATE, strict=False)
missing = [k for k in missing if k not in ("lm_head.weight", "transformer.wte.weight")]
assert not unexpected and not missing, f"state-dict mismatch | missing: {missing[:5]} | unexpected: {list(unexpected)[:5]}"
assert base.lm_head.weight.data_ptr() == base.transformer.wte.weight.data_ptr(), "weight tying lost"
print("Phase 1 weights loaded: all", len(PHASE1_STATE), "tensors matched.")
del PHASE1_STATE

base = base.to(device).eval()

# Check 1: a pretrained model must predict legal text far better than chance (random init ~ ln(32000) = 10.4)
losses = []
with torch.no_grad():
    for d in train_docs[:4]:
        flat = [t for sent in d["sent_tok"] for t in sent][:512]
        if len(flat) < 16:
            continue
        x = torch.tensor([flat], dtype=torch.long, device=device)
        _, loss = base(x[:, :-1].contiguous(), x[:, 1:].contiguous())
        losses.append(loss.item())
lm_loss = float(np.mean(losses))
print(f"Pretrained LM loss on training text: {lm_loss:.3f}   (random init would be about {math.log(cfg.vocab_size):.2f})")
assert lm_loss < 8.0, "LM loss is close to chance: the checkpoint did not load meaningfully."

# Check 2: LORMA injection must not change the model's function at initialisation
probe_doc = max(train_docs, key=lambda d: sum(len(t) for t in d["sent_tok"]))
probe_ids = [t for sent in probe_doc["sent_tok"] for t in sent][:512]
probe = torch.tensor([probe_ids], dtype=torch.long, device=device)
with torch.no_grad():
    ref_hidden = base._run(probe)[0].float().clone()

model = DecoderLORMAForRR(base, NUM_LABELS, LORMA_RANK, LORMA_ALPHA,
                          subtract_principal=SUBTRACT_PRINCIPAL, use_pos=USE_POSITION_FEATURES).to(device)
model.eval()
with torch.no_grad():
    new_hidden = model.base._run(probe)[0].float()
diff = (ref_hidden - new_hidden).abs().max().item()
print(f"Max hidden-state change caused by LORMA injection: {diff:.2e}")
if SUBTRACT_PRINCIPAL:
    assert diff < 1e-2, "LORMA injection altered the pretrained function."

trainable = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
n_train = sum(p.numel() for _, p in trainable)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_train/1e6:.3f}M of {n_total/1e6:.2f}M (LORMA adapters + classifier head)")
del ref_hidden, new_hidden, probe

In [ ]:
# =============================================================================
# Batching, prediction and evaluation
# =============================================================================
from sklearn.metrics import f1_score, accuracy_score, classification_report

def collate(batch):
    B = len(batch)
    T = max(len(w["input_ids"]) for w in batch)
    S = max(len(w["end_idx"]) for w in batch)
    ids = torch.zeros(B, T, dtype=torch.long)
    end = torch.zeros(B, S, dtype=torch.long)
    rel = torch.zeros(B, S, dtype=torch.float32)
    lab = torch.full((B, S), -1, dtype=torch.long)
    for i, w in enumerate(batch):
        t, s = len(w["input_ids"]), len(w["end_idx"])
        ids[i, :t] = w["input_ids"]; end[i, :s] = w["end_idx"]
        rel[i, :s] = w["rel"]; lab[i, :s] = w["labels"]
    return ids, end, rel, lab

@torch.no_grad()
def predict_doc(model, doc):
    # Returns one predicted label id per sentence of the document (length == number of sentences).
    model.eval()
    n = len(doc["sent_tok"])
    preds = [-1] * n
    for w in build_windows(doc, use_labels=False):
        ids, end, rel, _ = collate([w])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(ids.to(device), end.to(device), rel.to(device))
        p = logits[0, :len(w["end_idx"])].argmax(-1).tolist()
        for j, (nw, g) in enumerate(zip(w["is_new"].tolist(), w["sent_idx"].tolist())):
            if nw:
                preds[g] = p[j]
    assert len(preds) == n and all(0 <= p < NUM_LABELS for p in preds), f"incomplete prediction for {doc['id']}"
    return preds

def evaluate_docs(model, docs):
    y_true, y_pred = [], []
    for d in docs:
        p = predict_doc(model, d)
        assert len(p) == len(d["labels"])
        y_true.extend(d["labels"]); y_pred.extend(p)
    present = sorted(set(y_true))
    macro = f1_score(y_true, y_pred, labels=present, average="macro", zero_division=0)
    return macro, accuracy_score(y_true, y_pred), y_true, y_pred
print("Prediction helpers ready.")

In [ ]:
# =============================================================================
# Training
# =============================================================================
from torch.utils.data import DataLoader

train_windows = [w for d in train_docs for w in build_windows(d, use_labels=True)]
assert train_windows, "no training windows"
ACCUM = max(1, GLOBAL_BS // MICRO_BS)
steps_per_epoch = math.ceil(math.ceil(len(train_windows) / MICRO_BS) / ACCUM)
total_steps = EPOCHS * steps_per_epoch
print(f"{len(train_windows)} training windows | micro-batch {MICRO_BS} x accum {ACCUM} | "
      f"{steps_per_epoch} steps/epoch | {total_steps} steps total")

lorma_params = [p for n, p in trainable if not n.startswith("classifier")]
head_params = [p for n, p in trainable if n.startswith("classifier")]
opt = torch.optim.AdamW([
    {"params": lorma_params, "lr": LR_LORMA, "weight_decay": 0.0, "base_lr": LR_LORMA},
    {"params": head_params, "lr": LR_HEAD, "weight_decay": WD_HEAD, "base_lr": LR_HEAD},
], betas=(0.9, 0.99))
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
cw = class_weights.to(device)

def lr_factor(step):
    warm = max(1, int(WARMUP_FRAC * total_steps))
    if step < warm:
        return (step + 1) / warm
    prog = (step - warm) / max(1, total_steps - warm)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * prog))

best_f1, best_epoch, best_state = -1.0, -1, None
global_step = 0
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    g = torch.Generator(); g.manual_seed(SEED + epoch)
    loader = DataLoader(train_windows, batch_size=MICRO_BS, shuffle=True, generator=g, collate_fn=collate)
    model.train()
    opt.zero_grad(set_to_none=True)
    run_loss, run_n = 0.0, 0

    for i, (ids, end, rel, lab) in enumerate(loader):
        ids, end, rel, lab = ids.to(device), end.to(device), rel.to(device), lab.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(ids, end, rel)
        logits = logits.float()
        valid = lab != -1
        if not valid.any():
            continue
        ce = F.cross_entropy(logits[valid], lab[valid], weight=cw)
        loss = ce + LORMA_L1 * lorma_l1(model)
        scaler.scale(loss / ACCUM).backward()
        run_loss += ce.item(); run_n += 1

        if global_step == 0 and i == 0:
            assert all(p.grad is not None for p in lorma_params + head_params), "some trainable parameters received no gradient"

        if (i + 1) % ACCUM == 0 or (i + 1) == len(loader):
            for gr in opt.param_groups:
                gr["lr"] = gr["base_lr"] * lr_factor(global_step)
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(lorma_params + head_params, 1.0)
            scaler.step(opt); scaler.update()
            opt.zero_grad(set_to_none=True)
            global_step += 1

    macro, acc, _, _ = evaluate_docs(model, val_docs)
    tag = ""
    if macro > best_f1:
        best_f1, best_epoch = macro, epoch
        best_state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
        torch.save(best_state, BEST_STATE_PATH)
        tag = "  <- best"
    print(f"epoch {epoch:2d}/{EPOCHS} | train loss {run_loss / max(1, run_n):.4f} | val macro-F1 {macro:.4f} | "
          f"val acc {acc:.4f} | {(time.time() - t0) / 60:.1f} min{tag}")

# restore the best epoch
with torch.no_grad():
    for n, p in model.named_parameters():
        if n in best_state:
            p.copy_(best_state[n].to(p.device))
print(f"\nRestored best epoch {best_epoch} (val macro-F1 {best_f1:.4f}). Adapter/head weights saved to {BEST_STATE_PATH}")

macro, acc, y_true, y_pred = evaluate_docs(model, val_docs)
present = sorted(set(y_true))
print(classification_report(y_true, y_pred, labels=present, target_names=[id_to_label[i] for i in present], zero_division=0))

In [ ]:
# =============================================================================
# Predict every sentence of every test document and write the submission
# =============================================================================
model.eval()
predictions = {}
t0 = time.time()
for n_done, d in enumerate(test_docs, 1):
    ids = predict_doc(model, d)
    assert len(ids) == d["n_sent"], f"{d['id']}: {len(ids)} predictions for {d['n_sent']} sentences"
    predictions[d["id"]] = [id_to_label[i] for i in ids]
    print(f"  {n_done}/{len(test_docs)} {d['id'][:60]} | {d['n_sent']} sentences", flush=True)
print(f"Done in {(time.time() - t0) / 60:.1f} min")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PREDICTION_PATH, arcname="prediction.json")
print("Wrote", PREDICTION_PATH, "and", SUBMISSION_ZIP, f"({os.path.getsize(SUBMISSION_ZIP) / 1024:.1f} KB)")

In [ ]:
# =============================================================================
# Independent validation of the zip you are about to submit
# (re-reads the raw test files and the zipped prediction.json from disk)
# =============================================================================
def validate_submission(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"zip must contain exactly prediction.json, found {names}"
        pred = json.loads(z.read("prediction.json").decode("utf-8"))
    assert isinstance(pred, dict), "prediction.json must be a {document_id: [labels]} object"

    expected, per_file = {}, {}
    for path in TEST_PATHS:
        docs = sents = 0
        for doc_id, rec in iter_records(load_json(path), path):
            n = len(get_sentences(rec, doc_id))
            assert doc_id not in expected, f"document id {doc_id} appears in more than one test file"
            expected[doc_id] = n
            docs += 1; sents += n
        per_file[os.path.basename(path)] = (docs, sents)

    missing = sorted(set(expected) - set(pred))
    extra = sorted(set(pred) - set(expected))
    bad_len, bad_label = [], []
    for doc_id, n in expected.items():
        p = pred.get(doc_id)
        if p is None:
            continue
        if not isinstance(p, list) or len(p) != n:
            bad_len.append((doc_id, n, len(p) if isinstance(p, list) else type(p).__name__))
        elif any((not isinstance(x, str)) or x not in label_to_id for x in p):
            bad_label.append(doc_id)

    for fn, (dn, sn) in per_file.items():
        print(f"  {fn}: {dn} documents, {sn} sentences")
    if missing or extra or bad_len or bad_label:
        raise AssertionError(f"INVALID SUBMISSION | missing: {missing} | unexpected: {extra} | "
                             f"wrong length (id, expected, got): {bad_len} | bad labels: {bad_label}")
    print(f"\nSUBMISSION READY: {len(pred)} documents, {sum(expected.values())} sentences, "
          f"one valid label per sentence. Upload {zip_path}")

validate_submission(SUBMISSION_ZIP)

The file to upload is `/kaggle/working/cs781_phase3/submission.zip` (it contains `prediction.json`). The last cell raises an error instead of printing `SUBMISSION READY` if any document is missing or any sentence count is off.